Import all the necessary libraries.

In [1]:
from bioio import BioImage          # for import of different formats (runs on bioformats)
import numpy as np                  # for mathematical operations
import napari                       # image viewer
import skimage                      # for operation with images, such as filtering or binary operations
import matplotlib.pyplot as plt     # for plotting

# Load data
Load data and show some information. 

In [2]:
filepath = r"C:\Users\sveco\Downloads\OneDrive_2026-01-15\Pics for Sholl analysis\20xWT2-29-4-25--ibaab5076_2025-05-05_Protocol 6_11.16.44_F05.ims"

img = BioImage(filepath)
raw = img.get_image_data().squeeze()    # squeeze removes the first dimension corresponding to time-points, as there is only one time point

print(img.metadata.images[0].pixels.physical_size_x)   # This shows the physical pixel size - depends on objective mag used

mip_both = np.max(raw, axis=1)
mip_green = mip_both[1]
print(mip_green.shape)    # Shows the number of pixels in the MIP image

0.6091666666666689
(1992, 2040)


# Image segmentation
## Preprocess green channel, binarize and convert to skeletons
Perform filtering on the green channel to supress noise and remove background.

In [3]:
blurred = skimage.filters.gaussian(mip_green, sigma=15, preserve_range=True)
dog = mip_green - blurred   # Difference of Gaussians, removes the background
median = skimage.filters.median(dog)    # Median filter to reduce noise

Find threshold using one of the automatic algorithms, binarize the image and then skeletonize.

In [13]:
threshold = skimage.filters.threshold_li(median)
# threshold = skimage.filters.threshold_isodata(sharpened)  # Alternative thresholding returning more extensive masks
thresholded = median > threshold

skeleton = skimage.morphology.skeletonize(thresholded)

viewer = napari.Viewer()
viewer.add_image(mip_green, name='second channel MIP', blending='additive')
viewer.add_image(skeleton, name='Skeleton', blending='additive', opacity=0.5, colormap='yellow')

<Image layer 'Skeleton' at 0x2450e0b7880>

## Extract somas and get astrocyte centers
Get mask of soma from the blurred green channel.

In [5]:
blurred_less = skimage.filters.gaussian(mip_green, sigma=4, preserve_range=True)
blurred_more = skimage.filters.gaussian(mip_green, sigma=10, preserve_range=True)
soma = blurred_less - blurred_more

threshold_soma = skimage.filters.threshold_triangle(soma)
soma_binary = soma > threshold_soma

Get mask of nuclei (not perfectly separated) from the DAPI channel. Not all nuclei belong to astrocytes (which are relevant for us).

In [6]:
nuclei_mip = mip_both[0]

median = skimage.filters.rank.median(nuclei_mip, skimage.morphology.disk(2))

threshold_nuclei = skimage.filters.threshold_triangle(median)
nuclei_binary = median > threshold_nuclei

c:\Users\sveco\miniforge3\envs\analysisPythonEnv\lib\site-packages\skimage\filters\rank\generic.py:332: UserWarning: Bad rank filter performance is expected due to a large number of bins (5916), equivalent to an approximate bitdepth of 12.5.
  image, footprint, out, mask, n_bins = _preprocess_input(


Combine the two masks to get only positive areas in both channels using the AND operation.

In [7]:
combined = soma_binary & nuclei_binary

Extract centroids of the soma areas (we set a size limit on the areas we want to consider).

In [8]:
size_limit = 5
labels = skimage.measure.label(combined, connectivity=2)

props = skimage.measure.regionprops(labels)
centroids = [prop.centroid for prop in props if prop.area > size_limit]

Create circles of defined radius around the centroid points.

In [9]:
radius = 7
offsets = np.array([[-radius, 0], [0, radius], [radius, 0], [0, -radius]])
circles = [np.array([[cy + dy, cx + dx] for dy, dx in offsets]) for cy, cx in centroids]

## Filter only valid skeletons and soma
Filter the skeletons to keep only the ones that overlap with the identified soma areas.

In [10]:
labels_skeleton = skimage.measure.label(skeleton, connectivity=2)
overlapping_labels = np.unique(labels_skeleton[(labels_skeleton > 0) & combined])
filtered_skeleton = np.isin(labels_skeleton, overlapping_labels)

Visualize the extracted skeletons.

In [11]:
viewer = napari.Viewer()
viewer.add_image(mip_green, name='second channel MIP', blending='additive')
viewer.add_image(filtered_skeleton, name='Filtered Skeleton', blending='additive', opacity=0.5, colormap='yellow')
viewer.add_points(centroids, size=5, face_color='red')
viewer.add_shapes(circles, shape_type='ellipse', face_color='transparent', edge_color='red')

<Shapes layer 'circles' at 0x2450cc8fdf0>